# Red Neuronal (MLP)

Entrena un `MLPClassifier` para predecir `imdb_clase` (calidad percibida): Bajo / Medio / Alto.

**Nota:** `imdb_rating` fue excluido del feature set para evitar leakage (es la fuente del target).
`rotten_tomatoes_score` tambien se excluye (mide calidad similar al target).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)


## 1. Cargar datos

In [ ]:
X_train = pd.read_csv('../data/processed/X_train_nn.csv')
X_test  = pd.read_csv('../data/processed/X_test_nn.csv')
y_train = pd.read_csv('../data/processed/y_train_nn.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test_nn.csv').squeeze()

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print('Balance train (0=Bajo, 1=Medio, 2=Alto):')
print(y_train.value_counts().sort_index())


## 2. Busqueda de hiperparametros (GridSearchCV)

Se optimiza F1-macro para tratar todas las clases por igual, independientemente de su frecuencia.

In [ ]:
param_grid = {
    'hidden_layer_sizes': [(64,), (128, 64), (128, 64, 32)],
    'activation': ['relu', 'tanh'],
    'alpha': [1e-4, 1e-3],
}

mlp = MLPClassifier(
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    max_iter=500,
    random_state=42,
)
grid = GridSearchCV(mlp, param_grid, scoring='f1_macro', cv=3, n_jobs=-1)
grid.fit(X_train, y_train)

print('Mejores parametros:', grid.best_params_)
print(f'Mejor F1-macro en CV: {grid.best_score_:.4f}')


In [ ]:
mejor_mlp = grid.best_estimator_
print(f'Iteraciones hasta convergencia: {mejor_mlp.n_iter_}')


## 3. Curva de entrenamiento

Se grafican la funcion de perdida (train) y el F1-macro de validacion en cada iteracion.
El early stopping detuvo el entrenamiento cuando la validacion dejo de mejorar.

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(mejor_mlp.loss_curve_, color='steelblue', label='Loss (train)')
ax1.set_xlabel('Iteracion')
ax1.set_ylabel('Loss', color='steelblue')

ax2 = ax1.twinx()
ax2.plot(mejor_mlp.validation_scores_, color='darkorange', label='F1-macro (val)')
ax2.set_ylabel('F1-macro', color='darkorange')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
ax1.set_title('Curva de entrenamiento - Red Neuronal')
plt.tight_layout()
plt.savefig('../reports/figures/16_loss_curve_mlp.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Calibracion (OVR)

Curvas de calibracion uno-vs-resto para cada clase.
Una curva bien calibrada sigue la diagonal: probabilidad predicha = frecuencia real.

In [ ]:
y_proba = mejor_mlp.predict_proba(X_test)
nombres_clases = ['Bajo', 'Medio', 'Alto']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (ax, nombre) in enumerate(zip(axes, nombres_clases)):
    y_bin = (y_test == i).astype(int)
    proba_clase = y_proba[:, i]
    bins = np.linspace(0, 1, 11)
    bin_idx = np.clip(np.digitize(proba_clase, bins) - 1, 0, 9)
    frac_pos = [y_bin[bin_idx == b].mean() if (bin_idx == b).sum() > 0 else float('nan') for b in range(10)]
    mean_pred = [proba_clase[bin_idx == b].mean() if (bin_idx == b).sum() > 0 else float('nan') for b in range(10)]
    mask = [not (mp != mp) for mp in mean_pred]
    ax.plot([mp for mp, m in zip(mean_pred, mask) if m],
            [fp for fp, m in zip(frac_pos, mask) if m], 'o-', label=f'Clase {nombre}')
    ax.plot([0, 1], [0, 1], 'k--', label='Perfecta')
    ax.set_xlabel('Probabilidad predicha')
    ax.set_ylabel('Fraccion positivos')
    ax.set_title(f'Calibracion - {nombre}')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../reports/figures/17_calibracion_mlp.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Metricas en test

**Hallazgo:** el MLP muestra F1-macro bajo (~0.36). Predecir la calidad percibida (imdb_clase) desde
metadata de plataforma es intrinsecamente mas dificil que predecir exito comercial (is_hit).
Las features disponibles no tienen alta correlacion con el rating de IMDb.
El modelo tiende a predecir la clase mayoritaria (Medio, 50% del dataset).

In [ ]:
y_pred = mejor_mlp.predict(X_test)
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f'F1-macro:    {f1_macro:.4f}')
print(f'F1-weighted: {f1_weighted:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=nombres_clases))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=nombres_clases).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de Confusion - Red Neuronal')
plt.tight_layout()
plt.savefig('../reports/figures/conf_matrix_mlp.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Guardar modelo

In [ ]:
joblib.dump(mejor_mlp, '../models/mlp.joblib')
print('Modelo guardado: models/mlp.joblib')
metricas_mlp = {
    'modelo': 'MLP',
    'f1_macro': round(f1_macro, 4),
    'f1_weighted': round(f1_weighted, 4),
    'hidden_layer_sizes': str(mejor_mlp.hidden_layer_sizes),
    'activation': mejor_mlp.activation,
    'alpha': mejor_mlp.alpha,
    'n_iter': mejor_mlp.n_iter_,
}
print(metricas_mlp)
